# Attributed ROI by Channel

**Goal:** Join attribution model outputs with ad spend data to calculate attributed ROI per channel, per attribution model (First-Touch, Last-Touch, Linear).

**Inputs:**
- `channel_attribution_summary.csv` (from `attribution_modeling.ipynb`)
- Ad spend data (campaign performance dataset, or global ads dataset — whichever holds `spend`/`cost` and `revenue` by channel)

**Output:** `channel_roi_summary.csv` — feeds the ROI visuals in the Power BI dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## 1. Load Data

- `attribution_path`: output from the attribution modeling notebook
- `spend_path`: the dataset containing channel-level spend and revenue (adjust filename/columns to match your schema)

In [ ]:
attribution_path = 'channel_attribution_summary.csv'
spend_path = '../data/marketing_campaign_performance_clean.csv'

attribution = pd.read_csv(attribution_path)
spend_raw = pd.read_csv(spend_path)

attribution.head()

## 2. Aggregate Spend and Revenue by Channel

Expected columns in the spend dataset: `channel`, `cost` (or `spend`), `revenue`.
Adjust the column names below if your dataset uses different labels.

In [ ]:
COST_COL = 'cost'
REVENUE_COL = 'revenue'

channel_spend = (
    spend_raw
    .groupby('channel')
    .agg(total_spend=(COST_COL, 'sum'), total_revenue=(REVENUE_COL, 'sum'))
    .reset_index()
)

channel_spend

## 3. Join Attribution Credit with Spend

For each model, attributed conversions are joined to channel-level spend. Revenue is allocated proportionally to each model's share of attributed conversions per channel, so revenue scales with how much credit that model assigns the channel.

In [ ]:
merged = attribution.merge(channel_spend, on='channel', how='left')

# Total attributed conversions per channel across all rows for that channel within a model
# (already aggregated per model+channel from the attribution notebook, so this is just the row value)
merged['attributed_revenue'] = (
    merged['attributed_conversions'] / merged.groupby('channel')['attributed_conversions'].transform('sum')
    * merged['total_revenue']
)

merged.head(10)

## 4. Calculate ROI per Channel per Model

ROI formula: `(Attributed Revenue − Spend) / Spend`

Spend is not split across models (the cost was incurred regardless of which attribution lens you use), so ROI here reflects how each model's revenue allocation compares against the same fixed spend.

In [ ]:
merged['roi'] = (merged['attributed_revenue'] - merged['total_spend']) / merged['total_spend']
merged['roi_pct'] = (merged['roi'] * 100).round(2)

channel_roi_summary = merged[[
    'model', 'channel', 'attributed_conversions',
    'total_spend', 'attributed_revenue', 'roi_pct'
]].sort_values(['model', 'roi_pct'], ascending=[True, False])

channel_roi_summary

## 5. Visualization: ROI by Channel and Model

In [ ]:
pivot = channel_roi_summary.pivot(index='channel', columns='model', values='roi_pct').fillna(0)
pivot = pivot[['first_touch', 'last_touch', 'linear']]

ax = pivot.plot(kind='bar', figsize=(10, 6))
plt.axhline(0, color='black', linewidth=0.8)
plt.title('ROI (%) by Channel and Attribution Model')
plt.ylabel('ROI (%)')
plt.xlabel('Channel')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Model')
plt.tight_layout()
plt.savefig('channel_roi_comparison.png', dpi=150)
plt.show()

## 6. Top and Bottom Performing Channels (Linear Model)

Linear is typically the most balanced model to use for headline ROI reporting, since it doesn't over-credit a single touchpoint.

In [ ]:
linear_roi = channel_roi_summary[channel_roi_summary['model'] == 'linear'].sort_values('roi_pct', ascending=False)

print('Top 3 channels by ROI (Linear model):')
print(linear_roi.head(3)[['channel', 'roi_pct']].to_string(index=False))

print('\nBottom 3 channels by ROI (Linear model):')
print(linear_roi.tail(3)[['channel', 'roi_pct']].to_string(index=False))

## 7. Export for Power BI

In [ ]:
channel_roi_summary.to_csv('channel_roi_summary.csv', index=False)
print('Exported: channel_roi_summary.csv')

## 8. Optional: Combined Attribution + ROI Table

If you'd rather hand Power BI a single flat file instead of two related tables,
`channel_roi_summary` already contains everything from `channel_attribution_summary`
(attributed_conversions came from that join in section 3), so no extra merge is
needed — just export it under one combined name.

In [ ]:
combined = channel_roi_summary.copy()
combined.to_csv('channel_attribution_roi_combined.csv', index=False)
print('Exported: channel_attribution_roi_combined.csv')
combined.head()

## Notes for the team

- `channel_roi_summary.csv` has one row per channel per model — filter by `model` in Power BI to let users toggle between First-Touch, Last-Touch, and Linear ROI views.
- Spend (`total_spend`) is the same across all three models for a given channel — only `attributed_revenue` and `roi_pct` change based on the attribution logic.
- If `COST_COL` / `REVENUE_COL` don't match your actual dataset's column names, update them in section 2 before rerunning.
- Recommend using the Linear model as the default/headline ROI view on the dashboard, with First-Touch and Last-Touch available as toggles for comparison.